# Imports

In [1]:
from cyvcf2 import VCF
import pandas as pd
import numpy as np
from scipy import stats

# Load Data

In [2]:
vcf = VCF("../data/ps3_gwas.vcf.gz")

In [3]:
phen = pd.read_csv("../data/ps3_gwas.phen", sep="\t", header=None)

In [4]:
pcs = pd.read_csv("../plink_results/ps3_gwas.eigenvec", sep=r"\s+", header=None)

# Clean and Align Phenotype Data

In [5]:
# Reorder phenotypes samples to match order of VCF samples
phen.columns = ["FID","IID","PHEN"]
phen = phen.set_index("IID")
samples = vcf.samples
y = phen.loc[samples]["PHEN"].values

# GWAS w/out PCA

In [ ]:
# Set up output file
with open("../python_results/gwas.linear", "w") as out:
    out.write("CHR\tBP\tSNP\tBETA\tP\tN\n")

    for v in vcf:
        # Convert genotype to dosage-like numeric
        g = v.gt_types.astype(float)
        missing_mask = (g == 2) # UNKNOWN
        g[g == 3] = 2 # HOM_ALT -> dosage 2
        g[missing_mask] = np.nan # UNKNOWN -> NaN

        # Mask wherever no phenotyupe or sample data
        mask = ~np.isnan(g) & ~np.isnan(y)
        g2 = g[mask]
        y2 = y[mask]

        # Skip if not enough samples
        if len(g2) < 3:
            continue

        # Calculate MAF aka p
        p = g2.mean() / 2

        # Flip allele to match PLINK using minor allele, not ALT allele as effect allele
        if p > 0.5:
            g2 = 2 - g2
            p = 1 - p

        # Don't include if MAF < 0.05
        if p < 0.05:
            continue

        # Regression on filtered samples
        y0 = y2 - y2.mean()
        g0 = g2 - g2.mean()
        dof = len(y2) - 2

        denom = g0 @ g0
        if denom == 0: # Skip monomorphic SNPs (only 1 allele present in all samples)
            continue

        beta = (g0 @ y0) / denom

        resid = y0 - beta * g0
        se = np.sqrt((resid @ resid) / dof / denom)

        t = beta / se
        pval = 2 * stats.t.sf(abs(t), dof)

        # Write and increment
        out.write(f"{v.CHROM}\t{v.POS}\t{v.ID}\t{beta}\t{pval}\t{len(y2)}\n")

# Clean and Align PC Data

In [7]:
# Reorder PCs to match order of VCF samples
pcs.columns = ["FID", "IID"] + [f"PC{i}" for i in range(1, pcs.shape[1] - 1)]
pcs = pcs.set_index("IID")
pc_matrix = pcs.loc[samples].drop(columns=["FID"]).values.astype(float)

# Add intercept/bias term (columns of 1s added to left side)
intercept = np.ones(len(pc_matrix))
C = np.column_stack([intercept, pc_matrix])

# Creates a boolean mask indicating which rows have no missing values in C precomputed for later use: (n_samples,)
covar_complete = ~np.any(np.isnan(C), axis=1)

# GWAS w/PCA

In [8]:
# Reload VCF
vcf = VCF("../data/ps3_gwas.vcf.gz")

In [9]:
# Compute y-residuals (removes the effect of the PCs and intercept from the phenotype)
covar_coefs_y = np.linalg.solve(C.T @ C, C.T @ y)
y_resid = y - C @ covar_coefs_y

In [ ]:
# Set up output file
with open("../python_results/gwas_covar.linear", "w") as out:
    out.write("CHR\tBP\tSNP\tBETA\tP\tN\n")

    for v in vcf:
        # Convert genotype to dosage-like numeric
        g = v.gt_types.astype(float)
        missing_mask = (g == 2) # UNKNOWN
        g[g == 3] = 2 # HOM_ALT -> dosage 2
        g[missing_mask] = np.nan # UNKNOWN -> NaN

        # Mask wherever no phenotyupe or sample data
        mask = ~np.isnan(g) & ~np.isnan(y_resid) & covar_complete
        g2 = g[mask]
        y2 = y_resid[mask]
        C2 = C[mask]

        # Skip if not enough samples
        if len(g2) < C2.shape[1] + 2:
            continue

        # Calculate MAF aka p
        p = g2.mean() / 2

        # Flip allele to match PLINK using minor allele, not ALT allele as effect allele
        if p > 0.5:
            g2 = 2 - g2
            p = 1 - p

        # Don't include if MAF < 0.05
        if p < 0.05:
            continue

        # Compute g-residuals (removes the effect of the PCs and intercept from the genotypes)
        covar_coefs_g = np.linalg.solve(C2.T @ C2, C2.T @ g2)
        g_resid = g2 - C2 @ covar_coefs_g 

        # Regression on filtered samples
        denom = g_resid @ g_resid
        if denom == 0: # Skip monomorphic SNPs (only 1 allele present in all samples)
            continue

        beta = (g_resid @ y2) / denom

        dof = len(y2) - C2.shape[1] - 1
        resid = y2 - beta * g_resid
        se = np.sqrt((resid @ resid) / dof / denom)

        t = beta / se
        pval = 2 * stats.t.sf(abs(t), dof)

        # Write and increment
        out.write(f"{v.CHROM}\t{v.POS}\t{v.ID}\t{beta}\t{pval}\t{len(y2)}\n")